# RNAshapes
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
method_name = "RNAshapes"
base = Path.cwd()

tools_dir = base.parent / 'tools'
tools_dir.mkdir(exist_ok=True)

In [ ]:
rnashapes_env = tools_dir / 'rnashapes'
if not rnashapes_env.exists():
    print("Creating RNAshapes conda environment...")
    !conda create --prefix {rnashapes_env} -c bioconda rnashapes -y > /dev/null
    print("RNAshapes environment created successfully")
else:
    print(f"RNAshapes environment already exists at {rnashapes_env}")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
prediction_dir = base.parent / 'prediction'
prediction_dir.mkdir(exist_ok=True)

In [ ]:
def run_folding(fasta_name, output_dir, temp=37):
  out_file_name = fasta_name + '.out'
  rnashapes_env = tools_dir / 'rnashapes'

  # make prediction and clean output file
  os.system(f"conda run -p {rnashapes_env} RNAshapes -mode mfe --temperature {temp} {fasta_name} 2>/dev/null | \
   sed 's/^ \\+//g' | \
   sed 's/ \\+/\\t/g' | \
   cut -f2 | \
   grep -v '^$' > {out_file_name}")

  final_output = str(output_dir / Path(out_file_name).name)
  os.system(f"cp {out_file_name} {final_output}")
  
  for tmp_file in [fasta_name, out_file_name]:
    if os.path.exists(tmp_file):
      os.remove(tmp_file)
  
  return final_output

In [ ]:
out_fasta_name = method_name
final_fasta_path = prediction_dir / (out_fasta_name + ".fasta")
if final_fasta_path.exists():
    final_fasta_path.unlink()

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")

for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    # Write a one-sequence fasta
    tmp_fasta = f'RNAshapes_tmp_{vid}.fasta'
    with open(tmp_fasta, "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    dot_file_name = run_folding(tmp_fasta, prediction_dir)
    elapsed_time = time.time() - start_time

    # Concatenate outputs
    os.system(f"cat {dot_file_name} >> {final_fasta_path}")
    print(f"{elapsed_time: .1f} s")

print(f"\nProcessing complete. Results saved to {final_fasta_path}")